# 🎯 Piper Tamil TTS Training

This notebook trains a Piper TTS model for Tamil. Run each cell in order.

**Requirements:**
- GPU VM (T4 or better)
- Your dataset folder containing:
  - `metadata.csv` (format: `filename|text`)
  - `wav_22050/` folder with WAV files at 22050 Hz
  - `dataset.conf` (configuration file)

## Step 1: Install Dependencies

In [ ]:
# Install system dependencies
!apt-get update && apt-get install -y espeak-ng build-essential

# Install Python dependencies
!pip install torch==1.13.1 torchvision==0.14.1 torchaudio==0.13.1 --extra-index-url https://download.pytorch.org/whl/cu117
!pip install pytorch-lightning==1.8.6
!pip install piper-phonemize onnxruntime librosa

print("✅ Dependencies installed!")

## Step 2: Clone and Build Piper

In [ ]:
import os

PIPER_DIR = "/content/piper"

if not os.path.exists(PIPER_DIR):
    !git clone https://github.com/rhasspy/piper.git {PIPER_DIR}
    %cd {PIPER_DIR}
    !git checkout a0f09cdf9155010a45c243bc8a4286b94f286ef4
    %cd src/python
    !pip install -e .
    !bash build_monotonic_align.sh
    print("✅ Piper built successfully!")
else:
    print("✅ Piper already exists")

# Add to path
import sys
sys.path.insert(0, f"{PIPER_DIR}/src/python")

## Step 3: Upload Your Dataset

**Option A: Upload ZIP file** (recommended)
- Create a ZIP of your `tamil_dataset` folder
- Run the cell below and upload the ZIP

**Option B: Manual upload** (skip this cell)
- Use the file browser to upload your dataset folder to `/content/dataset/`

In [ ]:
import os
import zipfile
from google.colab import files

DATASET_DIR = "/content/dataset"
os.makedirs(DATASET_DIR, exist_ok=True)

print("📤 Please upload your dataset ZIP file...")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f"Extracting {filename}...")
    with zipfile.ZipFile(filename, 'r') as zip_ref:
        zip_ref.extractall(DATASET_DIR)
    os.remove(filename)

# Show what was extracted
print("\n📁 Dataset contents:")
for item in os.listdir(DATASET_DIR):
    print(f"  - {item}")

# Find the actual dataset folder
if os.path.exists(f"{DATASET_DIR}/metadata.csv"):
    FINAL_DATASET = DATASET_DIR
else:
    # Look for subfolder
    for item in os.listdir(DATASET_DIR):
        if os.path.isdir(f"{DATASET_DIR}/{item}"):
            if os.path.exists(f"{DATASET_DIR}/{item}/metadata.csv"):
                FINAL_DATASET = f"{DATASET_DIR}/{item}"
                break
    else:
        FINAL_DATASET = DATASET_DIR

print(f"\n✅ Dataset folder: {FINAL_DATASET}")

## Step 4: Configure Dataset Path

**Set the correct path to your dataset below:**

In [ ]:
# ========================================
# EDIT THIS: Set your dataset path
# ========================================
DATASET_DIR = "/content/dataset/tamil_dataset"  # <- CHANGE THIS if needed

# Configuration
LANGUAGE = "ta"  # Tamil espeak-ng identifier
SAMPLE_RATE = 22050
QUALITY = "medium"  # x-low, medium, or high
BATCH_SIZE = 32  # Reduce to 16 if you get OOM errors
MAX_EPOCHS = 5000

# Output directories
OUTPUT_DIR = "/content/training_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verify dataset
required_files = ["metadata.csv"]
for f in required_files:
    path = os.path.join(DATASET_DIR, f)
    if os.path.exists(path):
        print(f"✅ Found: {f}")
    else:
        print(f"❌ Missing: {f}")

# Check for audio folder
wav_folder = None
for candidate in ["wav_22050", "wavs", "audio"]:
    if os.path.exists(os.path.join(DATASET_DIR, candidate)):
        wav_folder = candidate
        break

if wav_folder:
    wav_count = len([f for f in os.listdir(os.path.join(DATASET_DIR, wav_folder)) if f.endswith('.wav')])
    print(f"✅ Found {wav_count} WAV files in {wav_folder}/")
else:
    print("⚠️ Could not find audio folder (wav_22050, wavs, or audio)")

## Step 5: Preprocess Dataset

In [ ]:
import subprocess
import os

PIPER_PYTHON = f"{PIPER_DIR}/src/python"
os.environ["PYTHONPATH"] = f"{os.environ.get('PYTHONPATH', '')}:{PIPER_PYTHON}"

print("🔄 Preprocessing dataset...")
print(f"   Input: {DATASET_DIR}")
print(f"   Output: {OUTPUT_DIR}")
print(f"   Language: {LANGUAGE}")

cmd = [
    "python3", "-m", "piper_train.preprocess",
    "--language", LANGUAGE,
    "--input-dir", DATASET_DIR,
    "--output-dir", OUTPUT_DIR,
    "--dataset-format", "ljspeech",
    "--single-speaker",
    "--sample-rate", str(SAMPLE_RATE),
    "--max-workers", "4"
]

result = subprocess.run(cmd, cwd=PIPER_PYTHON, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("❌ Error:")
    print(result.stderr)
else:
    print("✅ Preprocessing complete!")
    print(f"\nOutput files:")
    for f in os.listdir(OUTPUT_DIR):
        print(f"  - {f}")

## Step 6: Start Training 🚀

This will take several hours. Monitor the loss values - they should decrease over time.

In [ ]:
import subprocess
import os

PIPER_PYTHON = f"{PIPER_DIR}/src/python"

print("🚀 Starting Piper Training...")
print(f"   Quality: {QUALITY}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Max Epochs: {MAX_EPOCHS}")
print("\n" + "="*50)

cmd = [
    "python3", "-m", "piper_train",
    "--dataset-dir", OUTPUT_DIR,
    "--accelerator", "gpu",
    "--devices", "1",
    "--batch-size", str(BATCH_SIZE),
    "--validation-split", "0.0",
    "--num-test-examples", "0",
    "--max_epochs", str(MAX_EPOCHS),
    "--checkpoint-epochs", "1",
    "--precision", "32",
    "--quality", QUALITY
]

# Run training (this will take hours)
process = subprocess.Popen(
    cmd,
    cwd=PIPER_PYTHON,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# Stream output
for line in iter(process.stdout.readline, ''):
    print(line, end='')
    
process.wait()
print("\n" + "="*50)
print("Training finished!")

## Step 7: Export Model to ONNX (After Training)

In [ ]:
import os
import glob

# Find the latest checkpoint
checkpoints = glob.glob(f"{OUTPUT_DIR}/lightning_logs/version_*/checkpoints/*.ckpt")
if checkpoints:
    latest_ckpt = max(checkpoints, key=os.path.getctime)
    print(f"Found checkpoint: {latest_ckpt}")
    
    # Export to ONNX
    ONNX_OUTPUT = "/content/tamil_voice.onnx"
    
    cmd = [
        "python3", "-m", "piper_train.export_onnx",
        latest_ckpt,
        ONNX_OUTPUT
    ]
    
    result = subprocess.run(cmd, cwd=PIPER_PYTHON, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode == 0:
        print(f"✅ Model exported to: {ONNX_OUTPUT}")
    else:
        print("❌ Export failed:")
        print(result.stderr)
else:
    print("❌ No checkpoints found. Training may not have completed.")

## Step 8: Test the Voice

In [ ]:
from IPython.display import Audio
import subprocess

# Test text (Tamil)
TEST_TEXT = "வணக்கம், இது ஒரு சோதனை."
TEST_OUTPUT = "/content/test_output.wav"

# Generate audio using piper
cmd = f'echo "{TEST_TEXT}" | piper --model {ONNX_OUTPUT} --output_file {TEST_OUTPUT}'
result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

if os.path.exists(TEST_OUTPUT):
    print("✅ Audio generated! Playing...")
    display(Audio(TEST_OUTPUT))
else:
    print("❌ Failed to generate audio")
    print(result.stderr)

## Step 9: Download Your Model

In [ ]:
from google.colab import files

if os.path.exists(ONNX_OUTPUT):
    files.download(ONNX_OUTPUT)
    print("📥 Download started!")
else:
    print("❌ No model file found to download")